# 📓 Semana 10 · Dia 4 — Feature Engineering in Unity Catalog

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition (offline) + 🔑 serving online (trial) |
| **Tempo estimado** | 2h |
| **Certificação alvo** | MLP (2026) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Feature table criada + consumida |

---


## 📖 Teoria — Feature Engineering in UC (2026)

Antes chamado **Feature Store**, agora é **Feature Engineering in Unity Catalog**: feature tables governadas no UC (catalog.schema.table), com linhagem, versionamento e reuso entre equipes.

- **Feature table**: tabela com `primary_keys` e colunas de features
- **Offline**: features usadas no treino (batch)
- **Online (Feature Serving)**: features servidas em tempo real na inferência (🔑 pago)

> 🎯 **Dica de prova (MLP 2026)**: 'qual recurso gerencia features?' → Feature Engineering in UC; a nomenclatura nova é cobrada.


### 💻 Na prática — Criando a feature table

Crie a feature table de calendário com o pacote `databricks-feature-engineering`.


In [ ]:
# Criar feature table no UC
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
from pyspark.sql.functions import dayofweek, month, year, dayofmonth
fe = FeatureEngineeringClient()

features_df = (spark.table("workspace.ouro.vendas_por_dia")
    .select("data_venda")
    .withColumn("dia_semana", dayofweek("data_venda"))
    .withColumn("dia_mes", dayofmonth("data_venda"))
    .withColumn("mes", month("data_venda"))
    .withColumn("ano", year("data_venda")))
fe.create_table(
    name="workspace.prata.features_calendario",
    primary_keys=["data_venda"],
    df=features_df,
    description="Features de calendário para previsão de receita")
print("Feature table criada: workspace.prata.features_calendario")

In [ ]:
# Reutilizar features no treino (point-in-time correto)
treino = fe.create_training_set(
    df=spark.table("workspace.ouro.vendas_por_dia").select("data_venda", "receita_total"),
    label="receita_total",
    feature_lookups=[
        FeatureLookup(table_name="workspace.prata.features_calendario",
                      lookup_key="data_venda")])
df_treino = treino.load_df().toPandas()
print("Treino com features do UC pronto:", df_treino.shape)

### 💻 Na prática — Servindo features online (trial)

No workspace pago: **Catalog → workspace.prata.features_calendario → Feature Serving → Create endpoint** — a tabela vira endpoint de baixa latência para inferência em tempo real.


In [ ]:
# Consumo online em produção (🔑)
print("""
1. Catalog > features_calendario > Feature Serving
2. Create endpoint (serverless)
3. O endpoint recebe primary_keys e retorna features:
   POST /serving-endpoints/features_calendario/invocations
   {"dataframe_records": [{"data_venda": "2024-11-20"}]}
""")
print("Feature Serving: inferência em tempo real com as mesmas features do treino.")

> 🎯 **Dica de prova**: MLP: feature table = tabela com primary_keys; training set com FeatureLookup evita data leakage; online serving é para inferência em tempo real.


## 🎯 Exercícios de fixação

**1.** Crie uma feature table com 2 colunas novas (ex.: media_receita_7d).

**2.** Por que usar FeatureLookup em vez de join manual?

**3.** O que é data leakage e como o training set evita?


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Feature nova

Calcule a média móvel de 7 dias no Spark e `fe.create_table(...)` com a nova coluna.

**2.** FeatureLookup

Garante point-in-time correctness (features do momento certo, sem vazar futuro) e versionamento/linhagem — join manual não faz isso.

**3.** Data leakage

Usar informação futura no treino (ex.: receita de amanhã). O training set alinha features ao timestamp da label — sem vazar.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*